# 初期化

* **Pygments**: ソースコードをシンタックスハイライトされたHTMLに変換するために使用します。  
* **tree\_sitter**: Pythonコードを解析し、構文木を生成するために使用します。  
* **tree\_sitter\_languages**: Tree-sitterが様々な言語のグラマーを利用できるようにするためのパッケージです (Pythonのグラマーもここから取得できます)。  
* **beautifulsoup4**: 生成されたHTMLを操作したり、メタ情報を追加したりするのに便利です (オプションですが、あると整形しやすいです)。

In [1]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
venv_path = "/content/drive/MyDrive/.venv"
!chmod +x {venv_path}/bin/python
!chmod +x {venv_path}/bin/pip
sys.path.append(f"{venv_path}/lib/python3.11/site-packages")

Mounted at /content/drive


In [ ]:
#!{venv_path}/bin/pip install   pytest ipytest

In [2]:
import tree_sitter_python as tspython
from tree_sitter import Language, Parser

PY_LANGUAGE = Language(tspython.language())
parser = Parser(PY_LANGUAGE)
tree = parser.parse(
    bytes(
        """
def foo():
    if bar:
        baz()
""",
        "utf8"
    )
)
src = bytes(
    """
def foo():
    if bar:
        baz()
""",
    "utf8",
)


def read_callable_byte_offset(byte_offset, point):
    return src[byte_offset : byte_offset + 1]


tree = parser.parse(read_callable_byte_offset, encoding="utf8")

src_lines = ["\n", "def foo():\n", "    if bar:\n", "        baz()\n"]


def read_callable_point(byte_offset, point):
    row, column = point
    if row >= len(src_lines) or column >= len(src_lines[row]):
        return None
    return src_lines[row][column:].encode("utf8")


tree = parser.parse(read_callable_point, encoding="utf8")

root_node = tree.root_node
assert root_node.type == 'module'
assert root_node.start_point == (1, 0)
assert root_node.end_point == (4, 0)

function_node = root_node.children[0]
assert function_node.type == 'function_definition'
assert function_node.child_by_field_name('name').type == 'identifier'

function_name_node = function_node.children[1]
assert function_name_node.type == 'identifier'
assert function_name_node.start_point == (1, 4)
assert function_name_node.end_point == (1, 7)

function_body_node = function_node.child_by_field_name("body")

if_statement_node = function_body_node.child(0)
assert if_statement_node.type == "if_statement"

function_call_node = if_statement_node.child_by_field_name("consequence").child(0).child(0)
assert function_call_node.type == "call"

function_call_name_node = function_call_node.child_by_field_name("function")
assert function_call_name_node.type == "identifier"

function_call_args_node = function_call_node.child_by_field_name("arguments")
assert function_call_args_node.type == "argument_list"


assert str(root_node) == (
    "(module "
        "(function_definition "
            "name: (identifier) "
            "parameters: (parameters) "
            "body: (block "
                "(if_statement "
                    "condition: (identifier) "
                    "consequence: (block "
                        "(expression_statement (call "
                            "function: (identifier) "
                            "arguments: (argument_list))))))))"
)

from sys import maxsize
from unittest import TestCase

from tree_sitter import Language, Query

import tree_sitter_html
import tree_sitter_javascript
import tree_sitter_json
import tree_sitter_python


class TestLanguage(TestCase):
    def setUp(self):
        self.html = tree_sitter_html.language()
        self.javascript = tree_sitter_javascript.language()
        self.json = tree_sitter_json.language()
        self.python = tree_sitter_python.language()

    def test_init_invalid(self):
        self.assertRaises(ValueError, Language, 42)

    def test_properties(self):
        lang = Language(self.python)
        self.assertEqual(lang.version, 14)
        self.assertEqual(lang.node_kind_count, 275)
        self.assertEqual(lang.parse_state_count, 2809)
        self.assertEqual(lang.field_count, 32)

    def test_node_kind_for_id(self):
        lang = Language(self.json)
        self.assertEqual(lang.node_kind_for_id(1), "{")
        self.assertEqual(lang.node_kind_for_id(3), "}")

    def test_id_for_node_kind(self):
        lang = Language(self.json)
        self.assertEqual(lang.id_for_node_kind(":", False), 4)
        self.assertEqual(lang.id_for_node_kind("string", True), 20)

    def test_node_kind_is_named(self):
        lang = Language(self.json)
        self.assertFalse(lang.node_kind_is_named(4))
        self.assertTrue(lang.node_kind_is_named(20))

    def test_node_kind_is_visible(self):
        lang = Language(self.json)
        self.assertTrue(lang.node_kind_is_visible(2))

    def test_field_name_for_id(self):
        lang = Language(self.json)
        self.assertEqual(lang.field_name_for_id(1), "key")
        self.assertEqual(lang.field_name_for_id(2), "value")

    def test_field_id_for_name(self):
        lang = Language(self.json)
        self.assertEqual(lang.field_id_for_name("key"), 1)
        self.assertEqual(lang.field_id_for_name("value"), 2)

    def test_next_state(self):
        lang = Language(self.javascript)
        self.assertNotEqual(lang.next_state(1, 1), 0)

    def test_lookahead_iterator(self):
        lang = Language(self.javascript)
        self.assertIsNotNone(lang.lookahead_iterator(0))
        self.assertIsNone(lang.lookahead_iterator(9999))

    def test_query(self):
        lang = Language(self.json)
        query = lang.query("(string) @string")
        self.assertIsInstance(query, Query)

    def test_eq(self):
        self.assertEqual(Language(self.json), Language(self.json))

    def test_hash(self):
        for name in ["html", "javascript", "json", "python"]:
            with self.subTest(language=name):
                lang = Language(getattr(self, name))
                hash_ = hash(lang) & maxsize << 1
                self.assertGreater(hash_, 0)



In [ ]:
import ipytest
ipytest.autoconfig()

def test_addition():
    assert 1 + 1 == 2

ipytest.run('-vv')


In [4]:
# 共有ドライブにアクセスして、そこに移動する

# Google Driveのマウント (コードに既に含まれていますが、念のため)
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

import os
from pathlib import Path

# 共有ドライブの名前と目的のディレクトリパスを設定
SHARED_DRIVE_NAME = "AISearch"  # ここを共有ドライブの実際の名前 に変更してください
TARGET_REPO_DIR_NAME = "copilot-web-demo" # 共有ドライブ内の目的のフォルダ名 に変更してください
TARGET_SUB_DIR_NAME = "test"        # 共有ドライブ内の目的のサブフォルダ名 に変更してください

# 共有ドライブのパスを構築
SHARED_DRIVE_PATH = Path(f'/content/drive/Shareddrives/AISearch')


# 例: /content/drive/Shared drives/My Shared Drive/github/test
TARGET_CURRENT_DIR  = SHARED_DRIVE_PATH  / "workspace"
TARGET_REPO_PATH    = SHARED_DRIVE_PATH / TARGET_REPO_DIR_NAME
TARGET_OUT_DIR_PATH = SHARED_DRIVE_PATH/ f"{TARGET_REPO_DIR_NAME}-html"
# 目的のディレクトリに移動
try:
    if TARGET_CURRENT_DIR.exists() and TARGET_CURRENT_DIR.is_dir():
        os.chdir(TARGET_CURRENT_DIR)
        print(f"Successfully changed current working directory to: {os.getcwd()}")
    else:
        print(f"Error: Neither {TARGET_CURRENT_DIR} nor {TARGET_REPO_PATH} is a valid directory. Cannot change directory.")

except FileNotFoundError:
    print(f"Error: The path '{TARGET_CURRENT_DIR}' or '{TARGET_REPO_PATH}' was not found.")
except Exception as e:
    print(f"An error occurred while changing directory: {e}")

# 現在の作業ディレクトリを確認
print(f"Current working directory after attempting to change: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Successfully changed current working directory to: /content/drive/Shareddrives/AISearch/workspace
Current working directory after attempting to change: /content/drive/Shareddrives/AISearch/workspace


In [ ]:

os.chdir(TARGET_REPO_PATH)
!git branch
!git checkout ai-studio-github-communication
!git remote -v
!git status
!git shortlog -sn --all

# Pythonスクリプトをパースして構造化する。

**注意点**:

* **Tree-sitterグラマーのパス**: Language.build\_library の部分は、あなたの環境でTree-sitterのPythonグラマーがどこにあるかに依存します。tree\_sitter\_languages パッケージを使うと、この部分が簡略化できることが多いです（スクリプトのif \_\_name\_\_ \== "\_\_main\_\_": 以下にその例を追加しました）。vendor/tree-sitter-python は、自分で tree-sitter-python リポジトリをクローンして vendor ディレクトリに置く場合の例です。  
* **要素抽出の精度**: extract\_python\_elements 関数内のTree-sitterクエリやノードトラバーサルは、非常に基本的な例です。より正確に関数やクラスの範囲、docstring、引数などを抽出するには、Tree-sitterのクエリ言語を学習し、より洗練されたクエリを書く必要があります。ネストされた関数やクラス、複雑なコード構造に対応するには改良が必要です。  
* **ファイル名**: 生成されるHTMLのファイル名は、元ファイル名、要素のタイプ、要素名から生成しています。衝突を避け、かつ分かりやすい命名規則を検討してください。  
* **エラーハンドリング**: このスクリプトには十分なエラーハンドリングが含まれていません。

Tree-sitterクエリのポイント:

Tree-sitterのクエリは非常に強力で、特定の構文パターンを正確に捉えることができます。例えば、関数定義だけでなく、その関数内の特定のアノテーションや、特定のAPI呼び出しパターンなども抽出対象にできます。最初はシンプルな関数/クラス定義から始め、徐々にクエリを洗練させていくと良いでしょう。

<html><head>
  <title>
   Function: test_clone_and_analyze (from test_utils.py)
  </title>
  <meta content="" name="description">
  <meta content="No docstring found." name="description">
  <meta content="function, test_clone_and_analyze, python, code, snippet" name="keywords">
  <meta content="test_utils.py" name="source-file">
  <meta content="function" name="element-type">
  <meta content="test_clone_and_analyze" name="element-name">
  <style>
   pre { line-height: 125%; }
td.linenos .normal { color: inherit; background-color: transparent; padding-left: 5px; padding-right: 5px; }
span.linenos { color: inherit; background-color: transparent; padding-left: 5px; padding-right: 5px; }
td.linenos .special { color: #000000; background-color: #ffffc0; padding-left: 5px; padding-right: 5px; }
span.linenos.special { color: #000000; background-color: #ffffc0; padding-left: 5px; padding-right: 5px; }
.highlight .hll { background-color: #ffffcc }
.highlight { background: #ffffff; }
.highlight .c { color: #888 } /* Comment */
.highlight .err { color: #F00; background-color: #FAA } /* Error */
.highlight .k { color: #080; font-weight: bold } /* Keyword */
.highlight .o { color: #333 } /* Operator */
.highlight .ch { color: #888 } /* Comment.Hashbang */
.highlight .cm { color: #888 } /* Comment.Multiline */
.highlight .cp { color: #579 } /* Comment.Preproc */
.highlight .cpf { color: #888 } /* Comment.PreprocFile */
.highlight .c1 { color: #888 } /* Comment.Single */
.highlight .cs { color: #C00; font-weight: bold } /* Comment.Special */
.highlight .gd { color: #A00000 } /* Generic.Deleted */
.highlight .ge { font-style: italic } /* Generic.Emph */
.highlight .ges { font-weight: bold; font-style: italic } /* Generic.EmphStrong */
.highlight .gr { color: #F00 } /* Generic.Error */
.highlight .gh { color: #000080; font-weight: bold } /* Generic.Heading */
.highlight .gi { color: #00A000 } /* Generic.Inserted */
.highlight .go { color: #888 } /* Generic.Output */
.highlight .gp { color: #C65D09; font-weight: bold } /* Generic.Prompt */
.highlight .gs { font-weight: bold } /* Generic.Strong */
.highlight .gu { color: #800080; font-weight: bold } /* Generic.Subheading */
.highlight .gt { color: #04D } /* Generic.Traceback */
.highlight .kc { color: #080; font-weight: bold } /* Keyword.Constant */
.highlight .kd { color: #080; font-weight: bold } /* Keyword.Declaration */
.highlight .kn { color: #080; font-weight: bold } /* Keyword.Namespace */
.highlight .kp { color: #038; font-weight: bold } /* Keyword.Pseudo */
.highlight .kr { color: #080; font-weight: bold } /* Keyword.Reserved */
.highlight .kt { color: #339; font-weight: bold } /* Keyword.Type */
.highlight .m { color: #60E; font-weight: bold } /* Literal.Number */
.highlight .s { background-color: #FFF0F0 } /* Literal.String */
.highlight .na { color: #00C } /* Name.Attribute */
.highlight .nb { color: #007020 } /* Name.Builtin */
.highlight .nc { color: #B06; font-weight: bold } /* Name.Class */
.highlight .no { color: #036; font-weight: bold } /* Name.Constant */
.highlight .nd { color: #555; font-weight: bold } /* Name.Decorator */
.highlight .ni { color: #800; font-weight: bold } /* Name.Entity */
.highlight .ne { color: #F00; font-weight: bold } /* Name.Exception */
.highlight .nf { color: #06B; font-weight: bold } /* Name.Function */
.highlight .nl { color: #970; font-weight: bold } /* Name.Label */
.highlight .nn { color: #0E84B5; font-weight: bold } /* Name.Namespace */
.highlight .nt { color: #070 } /* Name.Tag */
.highlight .nv { color: #963 } /* Name.Variable */
.highlight .ow { color: #000; font-weight: bold } /* Operator.Word */
.highlight .w { color: #BBB } /* Text.Whitespace */
.highlight .mb { color: #60E; font-weight: bold } /* Literal.Number.Bin */
.highlight .mf { color: #60E; font-weight: bold } /* Literal.Number.Float */
.highlight .mh { color: #058; font-weight: bold } /* Literal.Number.Hex */
.highlight .mi { color: #00D; font-weight: bold } /* Literal.Number.Integer */
.highlight .mo { color: #40E; font-weight: bold } /* Literal.Number.Oct */
.highlight .sa { background-color: #FFF0F0 } /* Literal.String.Affix */
.highlight .sb { background-color: #FFF0F0 } /* Literal.String.Backtick */
.highlight .sc { color: #04D } /* Literal.String.Char */
.highlight .dl { background-color: #FFF0F0 } /* Literal.String.Delimiter */
.highlight .sd { color: #D42 } /* Literal.String.Doc */
.highlight .s2 { background-color: #FFF0F0 } /* Literal.String.Double */
.highlight .se { color: #666; font-weight: bold; background-color: #FFF0F0 } /* Literal.String.Escape */
.highlight .sh { background-color: #FFF0F0 } /* Literal.String.Heredoc */
.highlight .si { background-color: #EEE } /* Literal.String.Interpol */
.highlight .sx { color: #D20; background-color: #FFF0F0 } /* Literal.String.Other */
.highlight .sr { color: #000; background-color: #FFF0FF } /* Literal.String.Regex */
.highlight .s1 { background-color: #FFF0F0 } /* Literal.String.Single */
.highlight .ss { color: #A60 } /* Literal.String.Symbol */
.highlight .bp { color: #007020 } /* Name.Builtin.Pseudo */
.highlight .fm { color: #06B; font-weight: bold } /* Name.Function.Magic */
.highlight .vc { color: #369 } /* Name.Variable.Class */
.highlight .vg { color: #D70; font-weight: bold } /* Name.Variable.Global */
.highlight .vi { color: #33B } /* Name.Variable.Instance */
.highlight .vm { color: #963 } /* Name.Variable.Magic */
.highlight .il { color: #00D; font-weight: bold } /* Literal.Number.Integer.Long */
  </style>
 </head>
 <body>
  <h1>
   Function: test_clone_and_analyze (from test_utils.py)
  </h1>
  <p>
   Docstring: No docstring found.
  </p>
  <div class="highlight">
   <table class="highlighttable">
    <tbody><tr>
     <td class="linenos">
      <div class="linenodiv">
       <pre><span class="normal"> 1</span>
<span class="normal"> 2</span>
<span class="normal"> 3</span>
<span class="normal"> 4</span>
<span class="normal"> 5</span>
<span class="normal"> 6</span>
<span class="normal"> 7</span>
<span class="normal"> 8</span>
<span class="normal"> 9</span>
<span class="normal">10</span>
<span class="normal">11</span></pre>
      </div>
     </td>
     <td class="code">
      <div>
       <pre><span></span><span class="k">def</span><span class="w"> </span><span class="nf">test_clone_and_analyze</span><span class="p">():</span>
    <span class="n">tmpdir</span> <span class="o">=</span> <span class="n">get_tmp_dir</span><span class="p">()</span>
    <span class="c1"># プロジェクトディレクトリの一つ上の階層にmswリポジトリがある前提でパスを取得</span>
    <span class="kn">from</span><span class="w"> </span><span class="nn">src.utils</span><span class="w"> </span><span class="kn">import</span> <span class="n">get_project_root</span>
    <span class="n">msw_repo_path</span> <span class="o">=</span> <span class="n">get_project_root</span><span class="p">()</span><span class="o">.</span><span class="n">parent</span> <span class="o">/</span> <span class="s2">"msw"</span>
    <span class="k">assert</span> <span class="n">msw_repo_path</span><span class="o">.</span><span class="n">exists</span><span class="p">()</span>
    <span class="n">files</span> <span class="o">=</span> <span class="n">list_files_recursive</span><span class="p">(</span><span class="n">msw_repo_path</span><span class="p">)</span>
    <span class="k">assert</span> <span class="s2">"README"</span> <span class="ow">in</span> <span class="s2">""</span><span class="o">.</span><span class="n">join</span><span class="p">(</span><span class="n">files</span><span class="p">)</span> <span class="ow">or</span> <span class="s2">"README.md"</span> <span class="ow">in</span> <span class="s2">""</span><span class="o">.</span><span class="n">join</span><span class="p">(</span><span class="n">files</span><span class="p">)</span>
    <span class="n">analysis</span> <span class="o">=</span> <span class="n">analyze_repo_files</span><span class="p">(</span><span class="n">msw_repo_path</span><span class="p">)</span>
    <span class="k">assert</span> <span class="nb">isinstance</span><span class="p">(</span><span class="n">analysis</span><span class="p">,</span> <span class="nb">list</span><span class="p">)</span>
    <span class="k">assert</span> <span class="nb">all</span><span class="p">(</span><span class="s2">"path"</span> <span class="ow">in</span> <span class="n">f</span> <span class="ow">and</span> <span class="s2">"lines"</span> <span class="ow">in</span> <span class="n">f</span> <span class="k">for</span> <span class="n">f</span> <span class="ow">in</span> <span class="n">analysis</span><span class="p">)</span>
</pre>
      </div>
     </td>
    </tr>
   </tbody></table>
  </div>


</body></html>

In [9]:
import os
from pygments import highlight
from pygments.lexers import get_lexer_by_name
from pygments.formatters import HtmlFormatter
from tree_sitter import Language, Parser

from bs4 import BeautifulSoup # オプション

# Tree-sitterのPythonグラマーをロード
# tree_sitter_languages をインストールした場合の一般的なロード方法
# Python以外の言語も同様にロード可能
#Language.build_library( 'build/my-languages.so', ['vendor/tree-sitter-python'])

import tree_sitter_python as tspython
from tree_sitter import Language, Parser





def extract_python_elements(source_code_path):
    """
    Pythonソースコードから関数とクラスを抽出する。
    将来的にはもっと詳細な情報を抽出できるように拡張する。
    """
    with open(source_code_path, 'r', encoding='utf-8') as f:
        code_string = f.read()

    tree = parser.parse(bytes(code_string, "utf8"))
    root_node = tree.root_node

    elements = []

    # Tree-sitterクエリを使って関数とクラスの定義を見つける
    # (より堅牢な方法はクエリを使うことです)
    # ここでは簡略化のため、トップレベルの関数/クラスを想定

    # 関数定義のクエリ例 (もっと洗練させる必要があります)
    function_query_str = """
    (function_definition
        name: (identifier) @function.name
        body: (block) @function.body)
    """
    # クラス定義のクエリ例
    class_query_str = """
    (class_definition
        name: (identifier) @class.name
        body: (block) @class.body)
    """

    query_function = PY_LANGUAGE.query(function_query_str)
    query_class = PY_LANGUAGE.query(class_query_str)

    captures_function = query_function.captures(root_node)
    captures_class = query_class.captures(root_node)

    # 抽出したノードを処理 (この部分は実際のコード構造に合わせて調整が必要)
    # 以下は非常に単純化した例

    # トップレベルの関数を探す (より良い方法はクエリを使う)
    for node in root_node.children:
        if node.type == 'function_definition':
            function_name_node = node.child_by_field_name('name')
            function_body_node = node.child_by_field_name('body')
            if function_name_node and function_body_node:
                element_name = function_name_node.text.decode('utf8')
                element_code = node.text.decode('utf8') # 関数全体のコード
                docstring_node = None
                # 簡単なdocstring抽出 (より堅牢な方法が必要)
                if function_body_node.children and \
                   function_body_node.children[0].type == 'expression_statement' and \
                   function_body_node.children[0].children[0].type == 'string':
                    docstring_node = function_body_node.children[0].children[0]

                docstring = docstring_node.text.decode('utf8').strip('"\'') if docstring_node else "No docstring found."

                elements.append({
                    "type": "function",
                    "name": element_name,
                    "code": element_code,
                    "docstring": docstring,
                    "source_file": os.path.basename(source_code_path)
                })

        elif node.type == 'class_definition':
            class_name_node = node.child_by_field_name('name')
            # クラスも同様に処理 (省略)
            if class_name_node:
                element_name = class_name_node.text.decode('utf8')
                element_code = node.text.decode('utf8')
                # docstringなども同様に
                elements.append({
                    "type": "class",
                    "name": element_name,
                    "code": element_code,
                    "docstring": "Class docstring placeholder", # TODO
                    "source_file": os.path.basename(source_code_path)
                })

    # ファイル全体も一つの要素として追加 (オプション)
    elements.append({
        "type": "file_summary",
        "name": os.path.basename(source_code_path),
        "code": code_string, # ファイル全体のコード
        "docstring": "Summary of the file.", # TODO: ファイルサマリーを生成
        "source_file": os.path.basename(source_code_path)
    })

    return elements

def create_html_from_element(element_info, output_dir, formatter, lexer):
    """
    抽出された要素情報からHTMLファイルを生成する。
    """
    html_code_block = highlight(element_info["code"], lexer, formatter)

    # BeautifulSoupを使って基本的なHTML構造を作成 (オプション)
    soup = BeautifulSoup("<html><head><title></title><meta name='description' content=''></head><body></body></html>", "html.parser")

    title_str = f"{element_info['type'].capitalize()}: {element_info['name']} (from {element_info['source_file']})"
    soup.title.string = title_str

    # メタ情報の設定 (例)
    meta_desc = soup.new_tag("meta", attrs={"name":"description", "content":element_info["docstring"][:]}) # descriptionは短く
    meta_keywords = soup.new_tag("meta", attrs={"name":"keywords", "content":f"{element_info['type']}, {element_info['name']}, python, code, snippet"}) # 簡単なキーワード
    meta_source_file = soup.new_tag("meta", attrs={"name":"source-file", "content":element_info['source_file']})
    meta_element_type = soup.new_tag("meta", attrs={"name":"element-type", "content":element_info['type']})
    meta_element_name = soup.new_tag("meta", attrs={"name":"element-name", "content":element_info['name']})

    soup.head.append(meta_desc)
    soup.head.append(meta_keywords)
    soup.head.append(meta_source_file)
    soup.head.append(meta_element_type)
    soup.head.append(meta_element_name)

    # Pygmentsが出力するCSSを埋め込む (オプション、外部CSSでも可)
    style_tag = soup.new_tag("style")
    style_tag.string = formatter.get_style_defs('.highlight') # .highlight はPygmentsのデフォルトクラス
    soup.head.append(style_tag)

    # HTMLのbodyに情報を追加
    header_tag = soup.new_tag("h1")
    header_tag.string = title_str
    soup.body.append(header_tag)

    docstring_p = soup.new_tag("p")
    docstring_p.string = f"Docstring: {element_info['docstring']}"
    soup.body.append(docstring_p)

    # ハイライトされたコードブロックをBeautifulSoupオブジェクトに変換して追加
    code_soup = BeautifulSoup(html_code_block, "html.parser")
    soup.body.append(code_soup)

    # HTMLファイルとして保存
    # ファイル名はユニークにする (例: sourcefile_type_name.html)
    # サニタイズ処理も忘れずに
    safe_element_name = "".join(c if c.isalnum() else "_" for c in element_info['name'])
    safe_source_file = os.path.splitext(element_info['source_file'])[0]
    output_filename = f"{safe_source_file}_{element_info['type']}_{safe_element_name}.html"
    output_path = os.path.join(output_dir, output_filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(str(soup.prettify()))
    print(f"Generated HTML: {output_path}")

def main():

    source_dir = TARGET_REPO_PATH  # ステップ1で準備したファイルがあるディレクトリ
    output_html_dir = TARGET_OUT_DIR_PATH # 生成されたHTMLを保存するディレクトリ

    if not os.path.exists(output_html_dir):
        os.makedirs(output_html_dir)

    # Pygmentsのレクサーとフォーマッターを準備
    python_lexer = get_lexer_by_name("python", stripall=True)
    # CSSをHTMLに埋め込むスタイル。linenos=Trueで行番号も
    html_formatter = HtmlFormatter(full=False, style="colorful", cssclass="highlight", linenos=True)

    print(f"Processing {source_dir}...  {os.listdir(source_dir)}")
    for root, _, files in os.walk(source_dir):
      for filename in files:
          if filename.endswith(".py"): # Pythonファイルのみ対象
              file_path = os.path.join(root, filename)
              print(f"Processing {file_path}...")
              relative_path = os.path.relpath(file_path, source_dir)

              # 出力ディレクトリ内での対応するディレクトリパスを作成
              # 例: output_base_dir = '/output', relative_path = 'subdir/file.py'
              # output_dir_for_file = '/output/subdir'
              output_dir_for_file = os.path.join(output_html_dir, os.path.dirname(relative_path))

              # 出力ディレクトリが存在しない場合は作成
              if not os.path.exists(output_dir_for_file):
                  os.makedirs(output_dir_for_file)

              elements = extract_python_elements(file_path)

              for element in elements:
                  create_html_from_element(element,output_dir_for_file, html_formatter, python_lexer)

if __name__ == "__main__":
    # Tree-sitterグラマーのビルドパス設定 (重要)
    # 環境に合わせて 'vendor/tree-sitter-python' などのパスを調整してください。
    # これは tree_sitter_languages を使ってより簡単に解決できる場合があります。
    # e.g., from tree_sitter_languages import get_language, get_parser
    # PY_LANGUAGE = get_language('python')
    # parser = get_parser('python')
    # parser.set_language(PY_LANGUAGE) # get_parserの場合は不要なことも

    # tree_sitter_languagesを使う場合の例 (より推奨):
    try:

        PY_LANGUAGE = Language(tspython.language())
        parser = Parser(PY_LANGUAGE)
        print("Tree-sitter Python grammar loaded successfully via tree_sitter_languages.")
    except Exception as e:
        print(f"Error loading tree-sitter grammar: {e}")
        print("Please ensure 'tree_sitter' and 'tree_sitter_languages' are installed and configured correctly.")
        print("You might need to clone specific grammar repos (e.g., tree-sitter-python) into a 'vendor' directory and adjust Language.build_library path if not using tree_sitter_languages properly.")
        exit()

    main()


Tree-sitter Python grammar loaded successfully via tree_sitter_languages.
Processing /content/drive/Shareddrives/AISearch/copilot-web-demo...  ['.git', '.github', 'docs', '.devcontainer', '.vscode', 'Dockerfile', 'index-dev.html', 'index.html', 'metadata.json', 'package-lock.json', 'package.json', 'public', 'sample.ipynb', 'src', 'static', 'test', 'tmp', 'tsconfig.json', 'vite.config.ts', '.gitignore', 'requirements.txt', 'README.md']
Processing /content/drive/Shareddrives/AISearch/copilot-web-demo/src/__init__.py...
Generated HTML: /content/drive/Shareddrives/AISearch/copilot-web-demo-html/src/__init___file_summary___init___py.html
Processing /content/drive/Shareddrives/AISearch/copilot-web-demo/src/ngrok_setup.py...
Generated HTML: /content/drive/Shareddrives/AISearch/copilot-web-demo-html/src/ngrok_setup_function_start_ngrok.html
Generated HTML: /content/drive/Shareddrives/AISearch/copilot-web-demo-html/src/ngrok_setup_file_summary_ngrok_setup_py.html
Processing /content/drive/Share

**特に重要な検証ポイント (再掲)**:

* **メタデータの認識**: あなたがHTMLファイル内に埋め込んだ\<meta name="element-type" content="..."\> のようなカスタムメタタグが、Vertex AI Searchによって正しく解釈され、検索時のフィルタリングやファセットとして利用できるか。スクリーンショットではファイルのアイコンと名前が認識されていますが、HTMLの内部構造（特にメタタグ）まで活用できるかが、このアプローチの価値を大きく左右します。  
* **検索対象となるコンテンツ**: HTMLの本文だけでなく、\<title\> タグや主要な見出しタグの内容も検索対象として適切に重み付けされているか。